# Modelos de Machine Learning
## Solución Inteligente de Seguridad Ciudadana para Santander

**Objetivo:** Implementar un sistema híbrido de predicción de criminalidad que integra tres módulos complementarios:

1. **Módulo Temporal**: Predicción de tendencias y patrones temporales
2. **Módulo Espacial**: Identificación de hotspots y análisis geográfico
3. **Módulo de Riesgo**: Clasificación y scoring de riesgo por zona-tiempo

**Arquitectura del Sistema:**
```
┌─────────────────────────────────────────────────────────┐
│          SISTEMA INTEGRADO DE PREDICCIÓN                │
└─────────────────────────────────────────────────────────┘
                          │
          ┌───────────────┼───────────────┐
          │               │               │
          ▼               ▼               ▼
    ┌─────────┐    ┌──────────┐    ┌─────────┐
    │ Módulo  │    │  Módulo  │    │ Módulo  │
    │Temporal │    │ Espacial │    │  Riesgo │
    └─────────┘    └──────────┘    └─────────┘
    SARIMA +       DBSCAN +         XGBoost
    Prophet        KDE              Classifier
         │              │                │
         └──────────────┼────────────────┘
                        ▼
              ┌──────────────────┐
              │ ENSEMBLE FINAL   │
              │ (Integración)    │
              └──────────────────┘
```

**Fecha de desarrollo:** Noviembre 19, 2025
**Datasets base:** Resultados del EDA (eda.ipynb)

## 📋 Tabla de Contenido

### FASE 1: Preparación de Datos
1. [Configuración Inicial](#1-configuracion)
2. [Carga y Preprocesamiento de Datos](#2-carga-datos)
3. [Ingeniería de Features](#3-feature-engineering)

### FASE 2: Módulo Temporal
4. [Preparación de Series Temporales](#4-prep-temporal)
5. [Modelo SARIMA](#5-sarima)
6. [Modelo Prophet](#6-prophet)
7. [Evaluación Módulo Temporal](#7-eval-temporal)

### FASE 3: Módulo Espacial
8. [Preparación de Datos Geográficos](#8-prep-espacial)
9. [Clustering con DBSCAN](#9-dbscan)
10. [Mapas de Calor con KDE](#10-kde)
11. [Evaluación Módulo Espacial](#11-eval-espacial)

### FASE 4: Módulo de Riesgo
12. [Preparación de Features para Clasificación](#12-prep-riesgo)
13. [Modelo XGBoost](#13-xgboost)
14. [Optimización de Hiperparámetros](#14-optimizacion)
15. [Evaluación Módulo de Riesgo](#15-eval-riesgo)

### FASE 5: Integración y Deployment
16. [Sistema Ensemble](#16-ensemble)
17. [Validación del Sistema Completo](#17-validacion)
18. [Exportación de Modelos](#18-export)
19. [Dashboard de Resultados](#19-dashboard)

---

# FASE 1: PREPARACIÓN DE DATOS

---

## 1. Configuración Inicial e Importación de Librerías <a id='1-configuracion'></a>

In [ ]:
# ============================================================================
# LIBRERÍAS GENERALES
# ============================================================================
import pandas as pd
import numpy as np
import warnings
import os
import pickle
import json
from datetime import datetime, timedelta
from dotenv import load_dotenv

# ============================================================================
# VISUALIZACIÓN
# ============================================================================
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================================
# MODELOS DE SERIES TEMPORALES
# ============================================================================
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from prophet import Prophet

# ============================================================================
# MODELOS DE CLUSTERING Y ANÁLISIS ESPACIAL
# ============================================================================
from sklearn.cluster import DBSCAN, KMeans
from scipy.stats import gaussian_kde
from scipy.spatial.distance import cdist

# ============================================================================
# MODELOS DE CLASIFICACIÓN Y REGRESIÓN
# ============================================================================
from xgboost import XGBClassifier, XGBRegressor
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

# ============================================================================
# PREPROCESAMIENTO Y FEATURE ENGINEERING
# ============================================================================
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score, TimeSeriesSplit
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# ============================================================================
# MÉTRICAS DE EVALUACIÓN
# ============================================================================
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_curve, f1_score, accuracy_score, precision_score, recall_score
)

# ============================================================================
# MANEJO DE DESBALANCEO DE CLASES
# ============================================================================
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# ============================================================================
# CONFIGURACIÓN
# ============================================================================
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Cargar variables de entorno
load_dotenv()
APP_TOKEN = os.getenv('AppToken')

# Configuración de visualizaciones
FIGSIZE_LARGE = (16, 8)
FIGSIZE_MEDIUM = (12, 6)
FIGSIZE_SMALL = (10, 5)

# Semilla para reproducibilidad
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ Librerías importadas exitosamente")
print(f"✅ Numpy: {np.__version__}")
print(f"✅ Pandas: {pd.__version__}")
print(f"✅ Random State: {RANDOM_STATE}")

## 2. Carga y Preprocesamiento de Datos <a id='2-carga-datos'></a>

### 2.1. Extracción de Datos desde APIs

Reutilizamos las funciones del EDA para extraer los datos actualizados.

In [ ]:
# Configuración de datasets (mismo del EDA)
DATASETS = {
    'delitos_bucaramanga': {
        'id': '75fz-q98y',
        'name': '40 Delitos Bucaramanga',
        'scope': 'municipal',
        'filter': None,
        'url': 'https://www.datos.gov.co/resource/75fz-q98y.json'
    },
    'info_delictiva_bucaramanga': {
        'id': 'x46e-abhz',
        'name': '150 Información Delictiva Bucaramanga',
        'scope': 'municipal',
        'filter': None,
        'url': 'https://www.datos.gov.co/resource/x46e-abhz.json'
    },
    'delitos_sexuales': {
        'id': 'fpe5-yrmw',
        'name': 'Delitos Sexuales Nacional',
        'scope': 'nacional',
        'filter': "departamento='SANTANDER'",
        'url': 'https://www.datos.gov.co/resource/fpe5-yrmw.json'
    },
    'violencia_intrafamiliar': {
        'id': 'vuyt-mqpw',
        'name': 'Violencia Intrafamiliar Nacional',
        'scope': 'nacional',
        'filter': "departamento='SANTANDER'",
        'url': 'https://www.datos.gov.co/resource/vuyt-mqpw.json'
    },
    'hurto_modalidades': {
        'id': 'd4fr-sbn2',
        'name': 'Hurto por Modalidades Nacional',
        'scope': 'nacional',
        'filter': "departamento='SANTANDER'",
        'url': 'https://www.datos.gov.co/resource/d4fr-sbn2.json'
    }
}

print("✅ Configuración de datasets completada")
print(f"📊 Total de datasets: {len(DATASETS)}")

In [ ]:
# NOTA: Para acelerar el desarrollo, vamos a cargar datos desde cache si existen
# De lo contrario, extraemos desde la API

import requests

def fetch_soda_data(url, where_clause=None, limit=50000, app_token=None, max_records=None):
    """Extrae datos de la API SODA con paginación automática."""
    all_data = []
    offset = 0
    headers = {}
    
    if app_token:
        headers['X-App-Token'] = app_token
    
    print(f"Extrayendo datos de: {url}")
    
    while True:
        params = {
            '$limit': limit,
            '$offset': offset,
            '$order': ':id'
        }
        
        if where_clause:
            params['$where'] = where_clause
        
        try:
            response = requests.get(url, params=params, headers=headers, timeout=30)
            response.raise_for_status()
            batch = response.json()
            
            if not batch:
                break
            
            all_data.extend(batch)
            offset += limit
            print(f"  Registros extraídos: {len(all_data)}", end='\r')
            
            if max_records and len(all_data) >= max_records:
                all_data = all_data[:max_records]
                break
            
            if len(batch) < limit:
                break
                
        except requests.exceptions.RequestException as e:
            print(f"\n❌ Error: {e}")
            break
    
    print(f"\n✅ Total: {len(all_data)} registros")
    return pd.DataFrame(all_data)

print("✅ Función de extracción definida")

### 2.2. Extracción y Limpieza de Datos

**Estrategia:**
- Extraer datasets desde API
- Aplicar limpiezas críticas del EDA:
  - Eliminar duplicados
  - Limpiar coordenadas erróneas
  - Convertir tipos de datos
  - Crear variables temporales

In [ ]:
# TODO: Extraer datasets
# Por ahora, marcamos como pendiente la extracción
# Los datos se extraerán en la siguiente celda

print("⏳ Pendiente: Extracción de datasets desde API")
print("📝 Datasets a extraer:")
for key, config in DATASETS.items():
    print(f"  - {config['name']}")

---

# PLANIFICACIÓN DETALLADA DE LOS 3 MÓDULOS

---

## 🎯 MÓDULO 1: PREDICCIÓN TEMPORAL

### Objetivo
Predecir el número de delitos en periodos futuros (diario, semanal, mensual) usando modelos de series temporales.

### Datasets Prioritarios
- **Principal:** "150 Información Delictiva Bucaramanga" (fecha completa, alta calidad)
- **Secundario:** "40 Delitos Bucaramanga" (solo año-mes)
- **Complementarios:** Datasets nacionales filtrados por Santander

### Variables Target
1. **Conteo total de delitos** por periodo
2. **Conteo por tipo de delito** (hurto, violencia, delitos sexuales)
3. **Conteo por zona/barrio** (si se requiere granularidad espacial)

### Modelos a Implementar

#### 1. SARIMA (Seasonal ARIMA)
**Ventajas:**
- Maneja estacionalidad (patrones mensuales, semanales)
- Interpretable estadísticamente
- Baseline robusto para series temporales

**Hiperparámetros a optimizar:**
- `p`: Orden autoregresivo (AR)
- `d`: Orden de diferenciación
- `q`: Orden de media móvil (MA)
- `P`, `D`, `Q`, `s`: Componentes estacionales

**Pasos de implementación:**
```python
1. Agregación temporal (mensual/semanal/diaria)
2. Descomposición estacional (trend, seasonal, residual)
3. Test de estacionariedad (Augmented Dickey-Fuller)
4. Identificación de parámetros (ACF, PACF)
5. Ajuste del modelo SARIMA
6. Validación cruzada temporal
7. Pronóstico de N periodos adelante
```

#### 2. Prophet (Facebook)
**Ventajas:**
- Maneja múltiples estacionalidades (diaria, semanal, anual)
- Robusto a valores faltantes
- Fácil incorporación de festivos y eventos especiales

**Características clave:**
- Componente de tendencia (growth)
- Estacionalidad (yearly, weekly, daily)
- Festivos colombianos (Año Nuevo, Semana Santa, etc.)
- Changepoints automáticos

**Pasos de implementación:**
```python
1. Preparar DataFrame con columnas 'ds' (fecha) y 'y' (valor)
2. Configurar calendario de festivos colombianos
3. Ajustar modelo Prophet con estacionalidades
4. Validar con horizonte móvil (rolling forecast)
5. Generar pronósticos con intervalos de confianza
6. Análisis de componentes (trend, seasonality)
```

### Features Temporales Derivadas
```python
# Variables a crear:
- año, mes, trimestre, semestre
- día_semana (1=Lunes, 7=Domingo)
- fin_de_semana (0/1)
- festivo (0/1) - calendario colombiano
- semana_año (1-52)
- día_año (1-365)
- mes_sin (sin(2π × mes/12))  # Captura ciclicidad
- mes_cos (cos(2π × mes/12))
```

### Métricas de Evaluación
- **MAE** (Mean Absolute Error): Error promedio en unidades originales
- **RMSE** (Root Mean Squared Error): Penaliza errores grandes
- **MAPE** (Mean Absolute Percentage Error): Error porcentual
- **R²**: Varianza explicada

### Validación
**Time Series Split:**
- No usar validación cruzada aleatoria (viola dependencia temporal)
- Usar `TimeSeriesSplit` con múltiples folds
- Train: datos históricos | Test: periodo futuro

**Horizonte de predicción:**
- Corto plazo: 7 días (predicción operativa)
- Mediano plazo: 30 días (planificación mensual)
- Largo plazo: 90 días (estrategia trimestral)

### Visualizaciones
1. Serie temporal original vs predicción
2. Descomposición estacional (trend + seasonal + residual)
3. ACF/PACF para diagnóstico
4. Residuales del modelo (debe ser ruido blanco)
5. Forecast con intervalos de confianza (80%, 95%)
6. Comparación modelos (SARIMA vs Prophet)

---

## 🗺️ MÓDULO 2: ANÁLISIS ESPACIAL

### Objetivo
Identificar zonas de alta criminalidad (hotspots) y predecir probabilidad de delitos en ubicaciones geográficas.

### Datasets Prioritarios
- **Principal:** "150 Información Delictiva Bucaramanga" (coordenadas + barrio)
- **Secundario:** "40 Delitos Bucaramanga" (tras geocodificación)

### Variables de Entrada
- **Coordenadas:** latitud, longitud
- **Zona:** barrios_hecho, zona (urbana/rural)
- **Contexto temporal:** hora, día_semana, mes
- **Tipo de delito:** conducta, delito_solo

### Modelos a Implementar

#### 1. DBSCAN (Density-Based Spatial Clustering)
**Objetivo:** Identificar clusters espaciales de delitos (hotspots)

**Ventajas:**
- No requiere especificar número de clusters a priori
- Identifica clusters de forma arbitraria (no solo circulares)
- Detecta outliers (delitos aislados)
- Robusto a ruido

**Hiperparámetros:**
- `eps`: Radio máximo de vecindad (en km o grados)
- `min_samples`: Mínimo de delitos para formar cluster
- `metric`: Distancia (haversine para coordenadas geográficas)

**Pasos de implementación:**
```python
1. Filtrar registros con coordenadas válidas (95% del dataset)
2. Convertir coordenadas a radianes (para distancia haversine)
3. Normalizar coordenadas si es necesario
4. Ajustar DBSCAN con diferentes valores de eps
5. Identificar clusters (etiquetas >= 0)
6. Calcular centroides de cada cluster
7. Clasificar delitos como: core, border, noise
8. Validar clusters con métricas:
   - Silhouette Score
   - Davies-Bouldin Index
   - Análisis de densidad intra-cluster
```

**Interpretación:**
- **Cluster core:** Zona de alta densidad delictiva (prioridad alta)
- **Cluster border:** Zona periférica de hotspot
- **Noise (-1):** Delitos aislados, no forman patrón espacial

#### 2. Kernel Density Estimation (KDE)
**Objetivo:** Generar mapas de calor probabilísticos de criminalidad

**Ventajas:**
- Visualización intuitiva de concentración espacial
- Estimación de probabilidad de delito en cualquier punto (lat, lon)
- Suaviza datos puntuales para identificar patrones

**Parámetros:**
- `bandwidth`: Ancho de banda del kernel (controla suavizado)
- `kernel`: Tipo de kernel (gaussian, exponential, etc.)

**Pasos de implementación:**
```python
1. Extraer coordenadas (X=longitud, Y=latitud)
2. Crear grilla 2D sobre área de Bucaramanga
3. Ajustar KDE bidimensional
4. Evaluar densidad en cada punto de la grilla
5. Normalizar densidades a [0, 1] (probabilidad relativa)
6. Generar mapa de calor interactivo
7. Identificar top N zonas de mayor densidad
8. Estratificar por:
   - Tipo de delito
   - Rango horario (madrugada, día, noche)
   - Día de la semana
```

**Visualizaciones:**
```python
# Mapas de calor a generar:
1. Heatmap general (todos los delitos)
2. Heatmap por tipo de delito (hurto, violencia, etc.)
3. Heatmap por franja horaria
4. Heatmap comparativo (día vs noche)
5. Overlay con clusters de DBSCAN
6. Mapa interactivo con Plotly/Folium
```

### Features Espaciales Derivadas
```python
# Variables a crear:
- distancia_a_hotspot: Distancia al cluster más cercano (km)
- densidad_local: Número de delitos en radio de 500m
- cluster_id: ID del cluster DBSCAN asignado
- probabilidad_kde: Score KDE normalizado
- zona_riesgo: Categoría (alta, media, baja) basada en KDE
- barrio_encoded: Encoding del barrio
- coordenadas_normalizadas: Scaled lat/lon
```

### Geocodificación de Registros Faltantes
**Problema:** 4.71% de registros sin coordenadas en "40 Delitos Bucaramanga"

**Solución:**
```python
1. Agrupar registros válidos por barrio
2. Calcular centroide (latitud_media, longitud_media) por barrio
3. Añadir pequeña variación aleatoria (jitter) para evitar puntos idénticos
4. Imputar coordenadas faltantes con centroide del barrio
5. Marcar registros imputados con flag 'geocoded_imputed'
```

### Métricas de Evaluación
- **Silhouette Score**: Calidad de clusters (-1 a 1, >0.5 es bueno)
- **Número de clusters identificados**
- **Porcentaje de delitos en hotspots** (vs aislados)
- **Densidad promedio** en zonas de alto riesgo
- **Cobertura geográfica** de clusters (km²)

### Validación
- Comparar hotspots identificados con conocimiento local
- Validación temporal: ¿Los hotspots se mantienen en el tiempo?
- Estabilidad de clusters con submuestreo (bootstrap)

### Visualizaciones
1. Mapa de puntos de delitos (scatter geográfico)
2. Clusters DBSCAN coloreados por cluster_id
3. Mapa de calor KDE (heatmap 2D)
4. Mapa interactivo Plotly/Folium con:
   - Puntos de delitos
   - Contornos de densidad
   - Polígonos de barrios
   - Markers de centroides de hotspots
5. Comparación temporal de hotspots (antes vs ahora)

## ⚠️ MÓDULO 3: CLASIFICACIÓN Y SCORING DE RIESGO

### Objetivo
Predecir la probabilidad de ocurrencia de un delito en una zona-tiempo específica y clasificar el nivel de riesgo.

### Datasets Prioritarios
- **Principal:** "150 Información Delictiva Bucaramanga"
- **Integrando:** Features del Módulo Temporal + Espacial

### Problemas de ML a Resolver

#### Problema 1: Clasificación Binaria de Riesgo
**Target:** ¿Ocurrirá al menos 1 delito en zona X en el próximo periodo Y?
- Clase 0: Sin delitos
- Clase 1: Con delitos

#### Problema 2: Clasificación Multiclase de Nivel de Riesgo
**Target:** Nivel de riesgo de la zona-tiempo
- Clase 0: Riesgo Bajo (0-2 delitos)
- Clase 1: Riesgo Medio (3-5 delitos)
- Clase 2: Riesgo Alto (6+ delitos)

#### Problema 3: Regresión de Conteo
**Target:** Número exacto de delitos esperados en zona-tiempo

### Variables de Entrada (Features)

#### Features Temporales (del Módulo 1)
```python
- año, mes, trimestre, semestre
- día_semana (1-7)
- fin_de_semana (0/1)
- festivo (0/1)
- hora_dia (si disponible)
- semana_año
- tendencia_temporal: Promedio móvil últimos N días
- volatilidad: Desviación estándar últimos N días
```

#### Features Espaciales (del Módulo 2)
```python
- latitud, longitud (normalizadas)
- cluster_id: ID de hotspot DBSCAN
- densidad_kde: Score de densidad KDE
- distancia_a_hotspot: Distancia al cluster más cercano
- barrio_encoded: Barrio como categoría
- zona_riesgo: Alta/Media/Baja
- densidad_local: Delitos en radio de 500m
```

#### Features Contextuales
```python
- tipo_delito: Categoría del delito
- arma_medio: Presencia de arma (si/no)
- genero: Género predominante víctimas en zona
- grupo_etario: Grupo etario predominante
- movilidad_victima: A pie, vehículo, etc.
```

#### Features Agregadas Históricas
```python
- delitos_7d: Total delitos últimos 7 días en zona
- delitos_30d: Total delitos últimos 30 días en zona
- delitos_mismo_dia_semana: Histórico del mismo día de semana
- tasa_crecimiento: % cambio vs periodo anterior
- estacionalidad: Score de patrón estacional
```

### Modelo Principal: XGBoost Classifier

**Ventajas:**
- Excelente desempeño con datos tabulares
- Maneja variables categóricas y numéricas
- Importancia de features interpretable
- Robusto a outliers y valores faltantes
- Regularización L1/L2 incorporada

**Hiperparámetros a optimizar:**
```python
{
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2],
    'reg_alpha': [0, 0.1, 0.5, 1],  # L1 regularization
    'reg_lambda': [1, 1.5, 2]        # L2 regularization
}
```

**Estrategia de búsqueda:**
- Fase 1: `RandomizedSearchCV` (exploración amplia, 50 iteraciones)
- Fase 2: `GridSearchCV` (refinamiento en región óptima)

### Manejo de Desbalanceo de Clases

**Problema detectado:** Clases desbalanceadas (más días sin delitos que con delitos)

**Soluciones a implementar:**
```python
1. Class Weights:
   - scale_pos_weight = (negatives / positives)
   - Penaliza más errores en clase minoritaria

2. SMOTE (Synthetic Minority Over-sampling):
   - Genera ejemplos sintéticos de clase minoritaria
   - Aplicar solo en train set, no en test

3. Random Under-sampling:
   - Reduce clase mayoritaria
   - Combinar con SMOTE (SMOTE + Tomek Links)

4. Threshold Tuning:
   - Ajustar umbral de decisión (default 0.5)
   - Optimizar para maximizar F1-score o Recall
```

### Pipeline de Preprocesamiento
```python
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Columnas numéricas
numeric_features = ['latitud', 'longitud', 'densidad_kde', 
                   'delitos_7d', 'delitos_30d', ...]
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Columnas categóricas
categorical_features = ['barrio', 'tipo_delito', 'arma_medio', ...]
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='DESCONOCIDO')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combinar transformadores
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])
```

### Métricas de Evaluación

#### Para Clasificación Binaria:
- **Accuracy**: Porcentaje de predicciones correctas
- **Precision**: De los que predigo como riesgo, cuántos lo son
- **Recall**: De los que son riesgo, cuántos detecto
- **F1-Score**: Media armónica de Precision y Recall
- **ROC-AUC**: Área bajo curva ROC (capacidad discriminativa)
- **Confusion Matrix**: TP, TN, FP, FN

#### Para Clasificación Multiclase:
- **Macro F1**: Promedio F1 de todas las clases
- **Weighted F1**: F1 ponderado por tamaño de clase
- **Classification Report**: Precision, Recall, F1 por clase

#### Para Regresión:
- **MAE**: Error absoluto medio
- **RMSE**: Error cuadrático medio
- **R²**: Varianza explicada

### Validación y Generalización

**Estrategia de validación:**
```python
# Validación temporal (NO shuffle)
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

# Mantener orden temporal:
# Train: Mes 1-6 | Val: Mes 7
# Train: Mes 1-7 | Val: Mes 8
# ...
```

**Validación de estabilidad:**
- K-Fold Cross Validation (5 folds)
- Monitorear varianza de métricas entre folds
- Estabilidad de feature importance

### Interpretabilidad del Modelo

**Feature Importance:**
```python
# XGBoost native importance
xgb_model.feature_importances_

# SHAP (SHapley Additive exPlanations)
import shap
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

# Visualizaciones SHAP:
1. shap.summary_plot()  # Importancia global
2. shap.dependence_plot()  # Relación feature-target
3. shap.force_plot()  # Explicación individual
4. shap.waterfall_plot()  # Contribución de features
```

### Output del Modelo

**Para cada registro (zona-tiempo):**
```python
{
    'zona_id': 'Barrio X',
    'fecha': '2025-11-20',
    'hora': 14,
    'probabilidad_delito': 0.73,  # Prob(delito=1)
    'nivel_riesgo': 'ALTO',  # Clasificación
    'delitos_esperados': 4.2,  # Predicción regresión
    'confianza': 0.85,  # Confianza de la predicción
    'top_features': [  # Features más influyentes
        ('densidad_kde', 0.35),
        ('delitos_7d', 0.22),
        ('fin_de_semana', 0.15),
        ...
    ]
}
```

### Aplicaciones Operativas

1. **Dashboard de Patrullaje:**
   - Mapa de calor de probabilidad de riesgo
   - Top 10 zonas de mayor riesgo hoy
   - Recomendación de asignación de recursos

2. **Alertas Tempranas:**
   - Notificación cuando zona supera umbral de riesgo
   - Alertas de cambios bruscos en tendencia

3. **Planificación Estratégica:**
   - Identificar zonas que necesitan intervención
   - Evaluar efectividad de operativos

### Visualizaciones
1. ROC Curve (TPR vs FPR)
2. Precision-Recall Curve
3. Confusion Matrix (heatmap)
4. Feature Importance (top 20)
5. SHAP Summary Plot
6. Calibration Plot (predicted prob vs observed)
7. Mapa de predicciones (overlay con mapa real)
8. Comparación Predicho vs Real (time series)

---

## 🔄 INTEGRACIÓN DE LOS 3 MÓDULOS

### Arquitectura del Sistema Integrado

```
┌───────────────────────────────────────────────────────────────┐
│                    CAPA DE DATOS                              │
│  - API SODA (datos.gov.co)                                    │
│  - Limpieza y preprocesamiento                                │
│  - Feature engineering                                        │
└───────────────────────────────────────────────────────────────┘
                           │
                           ▼
┌───────────────────────────────────────────────────────────────┐
│                 MÓDULOS PREDICTIVOS                           │
│                                                               │
│  ┌─────────────┐  ┌──────────────┐  ┌─────────────────┐     │
│  │  TEMPORAL   │  │   ESPACIAL   │  │     RIESGO      │     │
│  ├─────────────┤  ├──────────────┤  ├─────────────────┤     │
│  │ • SARIMA    │  │ • DBSCAN     │  │ • XGBoost       │     │
│  │ • Prophet   │  │ • KDE        │  │ • Feature Imp   │     │
│  │             │  │ • Geocoding  │  │ • SHAP          │     │
│  └─────────────┘  └──────────────┘  └─────────────────┘     │
│         │                │                    │              │
└─────────┼────────────────┼────────────────────┼──────────────┘
          │                │                    │
          └────────────────┼────────────────────┘
                           ▼
┌───────────────────────────────────────────────────────────────┐
│              CAPA DE INTEGRACIÓN (ENSEMBLE)                   │
│                                                               │
│  Combina predicciones de los 3 módulos:                      │
│  - Predicción temporal → Tendencia esperada                   │
│  - Análisis espacial → Ubicación de riesgo                    │
│  - Score de riesgo → Probabilidad combinada                   │
│                                                               │
│  Output: Score final de riesgo (0-100) por zona-tiempo       │
└───────────────────────────────────────────────────────────────┘
                           │
                           ▼
┌───────────────────────────────────────────────────────────────┐
│               CAPA DE APLICACIÓN                              │
│                                                               │
│  • Dashboard interactivo (Streamlit/Dash)                     │
│  • API REST para consultas                                    │
│  • Sistema de alertas automáticas                             │
│  • Reportes ejecutivos                                        │
└───────────────────────────────────────────────────────────────┘
```

### Estrategia de Ensemble

**Opción 1: Votación Ponderada**
```python
score_final = (
    w1 * prediccion_temporal_normalizada +
    w2 * densidad_espacial_normalizada +
    w3 * probabilidad_xgboost
)

# Pesos iniciales (ajustar según validación):
w1 = 0.3  # Temporal
w2 = 0.3  # Espacial
w3 = 0.4  # Riesgo (más peso, modelo más completo)
```

**Opción 2: Stacking (Meta-modelo)**
```python
# Usar predicciones de módulos como features
X_meta = pd.DataFrame({
    'pred_sarima': predicciones_sarima,
    'pred_prophet': predicciones_prophet,
    'cluster_id': clusters_dbscan,
    'densidad_kde': scores_kde,
    'prob_xgboost': probabilidades_xgboost
})

# Meta-modelo (Regresión Logística o XGBoost)
meta_model = LogisticRegression()
meta_model.fit(X_meta, y_true)
```

### Flujo de Trabajo Completo

```python
# PASO 1: Preparación de Datos
df_raw = fetch_all_datasets()
df_clean = apply_cleaning_pipeline(df_raw)
df_features = engineer_features(df_clean)

# PASO 2: Módulo Temporal
# Agregar por periodo (diario/semanal/mensual)
ts_data = aggregate_temporal(df_features, freq='D')  # Diario

# Entrenar modelos
sarima_model = train_sarima(ts_data)
prophet_model = train_prophet(ts_data)

# Predicciones
forecast_sarima = sarima_model.forecast(steps=30)  # 30 días
forecast_prophet = prophet_model.predict(future_dates)

# PASO 3: Módulo Espacial
# Filtrar datos con coordenadas
df_geo = df_features[df_features['latitud'].notna()]

# Clustering
dbscan = train_dbscan(df_geo[['latitud', 'longitud']])
clusters = dbscan.labels_

# Density estimation
kde_model = train_kde(df_geo[['latitud', 'longitud']])
density_scores = evaluate_kde_on_grid(kde_model)

# PASO 4: Módulo de Riesgo
# Preparar features combinadas
X_risk = prepare_risk_features(
    df_features,
    temporal_trends=forecast_sarima,
    spatial_clusters=clusters,
    spatial_density=density_scores
)
y_risk = create_risk_labels(df_features, threshold=3)

# Entrenar XGBoost
xgb_model = train_xgboost(X_risk, y_risk)

# Predicciones
risk_predictions = xgb_model.predict_proba(X_risk)[:, 1]

# PASO 5: Integración (Ensemble)
final_scores = ensemble_predictions(
    temporal=forecast_prophet,
    spatial=density_scores,
    risk=risk_predictions,
    weights=[0.3, 0.3, 0.4]
)

# PASO 6: Generar Outputs
recommendations = generate_recommendations(final_scores)
alerts = generate_alerts(final_scores, threshold=0.7)

# PASO 7: Visualización
dashboard = create_dashboard(
    temporal_forecast=forecast_prophet,
    spatial_heatmap=density_scores,
    risk_map=final_scores
)
```

### Evaluación del Sistema Completo

**Métricas de negocio:**
- **Tasa de acierto en alertas**: % de alertas que resultaron en delitos reales
- **Cobertura de delitos predichos**: % de delitos que fueron anticipados
- **Tiempo de anticipación**: Días promedio de aviso previo
- **Eficiencia de recursos**: Reducción en patrullaje aleatorio

**Métricas técnicas:**
- **Error de predicción temporal**: MAE, RMSE del forecast
- **Calidad de clusters**: Silhouette score
- **Precisión de clasificación**: F1, ROC-AUC del XGBoost
- **Correlación ensemble**: Correlación entre score final y delitos reales

## 📊 RESUMEN EJECUTIVO DE LA PLANIFICACIÓN

### Datasets Utilizados

| Dataset | Uso Principal | Módulo |
|---------|---------------|--------|
| 150 Información Delictiva | Feature engineering completo | Todos |
| 40 Delitos Bucaramanga | Series temporales mensuales | Temporal |
| Delitos Sexuales | Análisis específico por tipo | Espacial, Riesgo |
| Violencia Intrafamiliar | Análisis específico por tipo | Espacial, Riesgo |
| Hurto Modalidades | Análisis de patrones de hurto | Todos |

### Modelos por Módulo

| Módulo | Modelos | Output | Métrica Principal |
|--------|---------|--------|-------------------|
| **Temporal** | SARIMA, Prophet | N° delitos en t+1 | MAE, RMSE |
| **Espacial** | DBSCAN, KDE | Hotspots, densidad | Silhouette, Visual |
| **Riesgo** | XGBoost | P(delito \| zona, tiempo) | F1, ROC-AUC |
| **Ensemble** | Voting/Stacking | Score final 0-100 | Correlación, Precisión |

### Variables Clave a Crear

**Temporales:**
- año, mes, trimestre, día_semana, fin_de_semana, festivo
- tendencia_7d, tendencia_30d, volatilidad
- mes_sin, mes_cos (ciclicidad)

**Espaciales:**
- cluster_id, densidad_kde, distancia_a_hotspot
- densidad_local (radio 500m)
- latitud_norm, longitud_norm

**Agregadas:**
- delitos_7d, delitos_30d (histórico)
- delitos_mismo_dia_semana
- tasa_crecimiento

### Limitaciones Identificadas

1. **Granularidad temporal heterogénea**
   - "40 Delitos": Solo año-mes (sin día)
   - Solución: Usar agregación mensual para ese dataset

2. **Coordenadas faltantes**
   - 4.71% sin coordenadas
   - Solución: Geocodificación por centroide de barrio

3. **Valores "NO REPORTA"**
   - Alta prevalencia en grupo etario (86%+)
   - Solución: Categoría explícita "NO_ESPECIFICADO"

4. **Desbalanceo de clases**
   - Más periodos sin delitos que con delitos
   - Solución: SMOTE, class weights, threshold tuning

### Entregables Esperados

1. **Modelos entrenados** (.pkl files)
   - sarima_model.pkl
   - prophet_model.pkl
   - dbscan_model.pkl
   - kde_model.pkl
   - xgboost_model.pkl
   - ensemble_model.pkl

2. **Notebooks documentados**
   - eda.ipynb ✅ (completado)
   - models.ipynb ⏳ (en desarrollo)
   - deployment.ipynb ⏳ (pendiente)

3. **Reportes**
   - Métricas de evaluación por modelo
   - Feature importance analysis
   - Validación temporal del sistema

4. **Visualizaciones**
   - Dashboards interactivos (Plotly)
   - Mapas de hotspots
   - Forecasts temporales
   - Matrices de confusión

### Próximos Pasos Inmediatos

**AHORA:** Comenzar implementación
1. Ejecutar extracción de datos
2. Aplicar pipeline de limpieza
3. Implementar feature engineering
4. Entrenar primer modelo (SARIMA baseline)

**¿Estás listo para comenzar con la implementación?**